In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# 1. Simpan OOF probability, fold, dan target agar eksperimen cepat tanpa retrain
oof_cols = [f"prob_{c}" for c in classes]
oof_df = pd.DataFrame(oof_prob, columns=oof_cols)
oof_df.insert(0, "fold", 0)
for f_num, (_, va_idx) in enumerate(skf.split(X, y), start=1):
    oof_df.loc[va_idx, "fold"] = f_num
oof_df["target"] = y

oof_df.to_parquet(RESULT_DIR / "pitri_oof_lgb.parquet", index=False)
print("OOF LightGBM tersimpan:", RESULT_DIR / "pitri_oof_lgb.parquet")


def fit_class_multipliers(probs, targets, n_classes, n_restarts=6, seed=42):
    """Cari pengali probabilitas per kelas untuk memaksimalkan Macro F1."""
    rng = np.random.default_rng(seed)
    counts = np.bincount(targets, minlength=n_classes).astype(np.float64)
    prior_boost = (counts.max() / np.maximum(counts, 1.0)) ** 0.30
    prior_boost /= prior_boost.mean()

    best_w = np.ones(n_classes, dtype=np.float64)
    best_score = f1_score(targets, np.argmax(probs, axis=1), average="macro")

    grid = np.linspace(0.35, 2.6, 46)
    init_list = [np.ones(n_classes), prior_boost]
    for _ in range(n_restarts):
        init_list.append(np.clip(prior_boost * rng.uniform(0.65, 1.35, size=n_classes), 0.35, 2.6))

    for w0 in init_list:
        w = w0.copy()
        cur_score = f1_score(targets, np.argmax(probs * w, axis=1), average="macro")
        improved = True
        while improved:
            improved = False
            for c in range(n_classes):
                best_val = w[c]
                for g in grid:
                    cand = w.copy()
                    cand[c] = g
                    sc = f1_score(targets, np.argmax(probs * cand, axis=1), average="macro")
                    if sc > cur_score + 1e-6:
                        cur_score = sc
                        best_val = g
                        improved = True
                w[c] = best_val
        if cur_score > best_score:
            best_score = cur_score
            best_w = w.copy()

    return best_w, best_score


# 2. Nested / out-of-fold multiplier tuning: fit di 4 fold, evaluasi di 1 fold sisanya
folds_arr = oof_df["fold"].to_numpy()
oof_tuned_pred = np.zeros(len(y), dtype=np.int32)
fold_ws = []

for f_val in range(1, N_SPLITS + 1):
    tr_mask = folds_arr != f_val
    va_mask = folds_arr == f_val
    w_fold, _ = fit_class_multipliers(
        oof_prob[tr_mask], y[tr_mask], N_CLASSES, n_restarts=4, seed=42 + f_val
    )
    fold_ws.append(w_fold)
    oof_tuned_pred[va_mask] = np.argmax(oof_prob[va_mask] * w_fold, axis=1)

raw_oof_f1 = f1_score(y, np.argmax(oof_prob, axis=1), average="macro")
tuned_oof_f1 = f1_score(y, oof_tuned_pred, average="macro")
w_final = np.mean(fold_ws, axis=0)

print(f"\nOOF Macro F1 (argmax)     : {raw_oof_f1:.6f}")
print(f"OOF Macro F1 (tuned OOF)  : {tuned_oof_f1:.6f}  (delta: {tuned_oof_f1 - raw_oof_f1:+.6f})")
print("Pengali rata-rata per kelas:", {str(classes[i]): round(float(w_final[i]), 4) for i in range(N_CLASSES)})

# 3. Simpan submission hasil tuning
sub_tuned = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: [classes[i] for i in np.argmax(test_prob * w_final, axis=1)],
})
out_tuned = RESULT_DIR / "submission_pitri_v2_tuned.csv"
sub_tuned.to_csv(out_tuned, index=False)
print("Submission v2 tersimpan:", out_tuned)

NameError: name 'classes' is not defined